# DeepDream: Visualizing Neural Network Perception

This notebook explores how a pretrained **InceptionV3** network responds to different visual patterns by applying **gradient ascent directly to the input image**.

The goal is not classification. Instead, the image is iteratively modified to strengthen whatever internal features a chosen layer activates most strongly. Early layers tend to amplify textures and repeated geometry, while deeper layers begin to produce more object-like fragments.

**Layers used in this notebook**
- `mixed2` — shallow features such as geometric motifs and swirling texture
- `mixed5` — mid-level texture blends and partial shapes
- `mixed8` — deeper, more semantic fragments such as eyes, fur, and face-like structures

The notebook includes:
- a single-scale DeepDream run
- a multi-scale octave version
- a step-size comparison
- a layer-by-layer comparison
- output export for saved images and progression frames

In [ ]:
# Core libraries
import os
import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, HTML
import base64

import tensorflow as tf
from tensorflow.keras.applications import inception_v3


## 1. Load the pretrained feature extractor

The classification head is removed so the notebook can work directly with the convolutional feature maps.

In [ ]:
# Load InceptionV3 without the classification head.
# This exposes the feature extractor used for DeepDream.
base_model = inception_v3.InceptionV3(include_top=False, weights="imagenet")

# Default layers for shallow, mid, and deeper feature comparisons.
default_layer_names = ["mixed2", "mixed5", "mixed8"]

layers = [base_model.get_layer(name).output for name in default_layer_names]
dream_model = tf.keras.Model(inputs=base_model.input, outputs=layers)

print("Loaded layers:", default_layer_names)

## 2. Choose an input image

When the notebook runs in Google Colab, it prompts for an upload. If no image is provided, a simple synthetic fallback image is generated so the rest of the notebook still works.

In [ ]:
image_path = None

try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        image_path = next(iter(uploaded.keys()))
except ImportError:
    # Notebook is running outside Colab.
    pass

In [ ]:
# Create a fallback image so the notebook can still run without a manual upload.
if image_path is None:
    w, h = 800, 500
    arr = np.zeros((h, w, 3), dtype=np.uint8)

    for y in range(h):
        arr[y, :, 0] = np.clip(40 + y // 8, 0, 255)
        arr[y, :, 1] = np.clip(80 + y // 6, 0, 255)
        arr[y, :, 2] = np.clip(140 + y // 4, 0, 255)

    arr[int(h * 0.65):, :] = [30, 110, 40]

    arr[120:350, 120:280] = [160, 160, 170]
    arr[140:330, 320:500] = [180, 170, 160]

    for x in range(135, 270, 35):
        for y in range(140, 320, 45):
            arr[y:y + 25, x:x + 20] = [40, 60, 100]

    for x in range(340, 490, 40):
        for y in range(155, 325, 50):
            arr[y:y + 28, x:x + 22] = [70, 90, 130]

    image_path = "synthetic_demo_image.jpg"
    Image.fromarray(arr).save(image_path)
    print("Created fallback demo image:", image_path)

In [ ]:
original_img = Image.open(image_path).convert("RGB")
display(original_img)
print("Original size:", original_img.size)

## 3. Utility functions

These helpers handle preprocessing, deprocessing, and quick visualization.

In [ ]:
def load_img(path, max_dim=800):
    """Load an image, resize it if needed, and preprocess it for InceptionV3."""
    img = Image.open(path).convert("RGB")
    max_side = max(img.size)
    if max_side > max_dim:
        scale = max_dim / max_side
        new_size = tuple(int(x * scale) for x in img.size)
        img = img.resize(new_size, Image.LANCZOS)
    img = np.array(img)
    img = inception_v3.preprocess_input(img.astype(np.float32))
    return tf.convert_to_tensor(img[None, ...])


def deprocess(img_tensor):
    """Convert an InceptionV3 tensor back to a displayable PIL image."""
    img = img_tensor.numpy()
    if len(img.shape) == 4:
        img = img[0]
    img = img.copy()
    img /= 2.0
    img += 0.5
    img *= 255.0
    return Image.fromarray(np.clip(img, 0, 255).astype(np.uint8))


def show_tensor_image(img_tensor, title=None, figsize=(12, 7)):
    plt.figure(figsize=figsize)
    plt.imshow(deprocess(img_tensor))
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()

## 4. Objective

DeepDream uses gradient ascent on the input image. The image is updated in the direction that increases the chosen layer activations.

The optimization target is:

- maximize `J(I)`

with the update rule:

- `I ← I + α · ∇ᵢ J(I)`

where `α` is the step size.

In [ ]:
# Optional visual summary of the optimization objective and update rule.
display(HTML("""
<div style="
    font-family: monospace;
    font-size: 28px;
    line-height: 2.2;
    padding: 40px 60px;
    background: #0d0d0d;
    color: #f0f0f0;
    border-radius: 12px;
    display: inline-block;
    margin-top: 10px;
">
    <div style="color:#aaa; font-size:16px; margin-bottom:16px;">objective</div>
    maximize &nbsp; <span style="color:#7ecfff;">J(I)</span>
    <br><br>
    <div style="color:#aaa; font-size:16px; margin-bottom:16px;">update rule</div>
    <span style="color:#7ecfff;">I</span>
    &nbsp;&#x2190;&nbsp;
    <span style="color:#7ecfff;">I</span>
    &nbsp;+&nbsp;
    <span style="color:#ffd97d;">&alpha;</span>
    &nbsp;&middot;&nbsp;
    <span style="color:#ffd97d;">&nabla;<sub style='font-size:18px'>I</sub></span>
    &nbsp;<span style="color:#7ecfff;">J(I)</span>
</div>
"""))

## 5. Build the loss and gradient step

The loss is the weighted sum of activations across the selected layers. One gradient ascent step then updates the image and clips it back into the valid InceptionV3 input range.

In [ ]:
# Layer weights control how much each target layer contributes to the dream.
layer_weights = {
    "mixed2": 0.6,
    "mixed5": 1.0,
    "mixed8": 1.4,
}

selected_layer_names = list(layer_weights.keys())
selected_outputs = [base_model.get_layer(name).output for name in selected_layer_names]
dream_model = tf.keras.Model(inputs=base_model.input, outputs=selected_outputs)


def calc_loss(img, model, layer_names, layer_weights_dict):
    """Compute the scalar objective J(I) to maximize."""
    img_batch = tf.expand_dims(img, axis=0) if len(img.shape) == 3 else img
    activations = model(img_batch)
    if not isinstance(activations, list):
        activations = [activations]

    losses = []
    for name, activation in zip(layer_names, activations):
        act = activation[:, 2:-2, 2:-2, :]
        losses.append(layer_weights_dict[name] * tf.reduce_mean(act))
    return tf.add_n(losses)

In [ ]:
@tf.function
def deepdream_step(img, step_size, model, layer_names, layer_weights_dict):
    """
    Run one gradient ascent step.

    Gradients are normalized so the step size stays stable across iterations.
    """
    with tf.GradientTape() as tape:
        tape.watch(img)
        loss = calc_loss(img, model, layer_names, layer_weights_dict)

    grads = tape.gradient(loss, img)
    grads /= tf.math.reduce_std(grads) + 1e-8
    img = img + grads * step_size
    img = tf.clip_by_value(img, -1.0, 1.0)
    return loss, img

## 6. Step-size sensitivity

This quick experiment shows why the learning rate matters. A very small step barely changes the image, while an overly large step tends to introduce unstable artifacts.

In [ ]:
def quick_dream(img, step_size, steps=40):
    img = tf.identity(img)
    names = ["mixed5"]
    outputs = [base_model.get_layer(n).output for n in names]
    model = tf.keras.Model(inputs=base_model.input, outputs=outputs)
    weights = {"mixed5": 1.0}
    for _ in range(steps):
        _, img = deepdream_step(img, step_size, model, names, weights)
    return img

img = load_img(image_path, max_dim=500)

dream_tiny  = quick_dream(img, step_size=0.001)
dream_good  = quick_dream(img, step_size=0.01)
dream_large = quick_dream(img, step_size=0.05)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, result, label in zip(
    axes,
    [dream_tiny, dream_good, dream_large],
    ["α = 0.001  (too small)", "α = 0.01   (balanced)", "α = 0.05   (too large)"],
):
    ax.imshow(deprocess(result))
    ax.set_title(label, fontsize=12)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. Single-scale DeepDream

This is the simplest version of the method: run gradient ascent repeatedly at one fixed resolution.

In [ ]:
def run_deepdream_simple(
    img,
    steps=80,
    step_size=0.01,
    model=dream_model,
    layer_names=selected_layer_names,
    layer_weights_dict=layer_weights,
):
    img = tf.identity(img)
    for step in range(steps):
        loss, img = deepdream_step(img, step_size, model, layer_names, layer_weights_dict)
        if (step + 1) % 20 == 0:
            print(f"Step {step + 1:>3} | Loss {loss.numpy():.4f}")
    return img


img = load_img(image_path, max_dim=700)
dream_img_simple = run_deepdream_simple(img, steps=80, step_size=0.01)
show_tensor_image(dream_img_simple, title="DeepDream (single scale)")

## 8. Multi-scale DeepDream with octaves

The octave version starts from a smaller representation and progressively works back up to larger resolutions. At each scale, detail lost during resizing is partially recovered from the original image.

In [ ]:
def run_deepdream_octaves(
    img,
    steps_per_octave=50,
    step_size=0.01,
    octave_scale=1.3,
    num_octaves=3,
    model=dream_model,
    layer_names=selected_layer_names,
    layer_weights_dict=layer_weights,
):
    """
    Run DeepDream across multiple image scales.

    Starting from smaller versions of the image usually produces richer detail
    than running the optimization only once at full resolution.
    """
    base_shape = tf.shape(img)[1:-1]
    float_base_shape = tf.cast(base_shape, tf.float32)

    octave_shapes = []
    for i in range(num_octaves):
        scale = octave_scale ** i
        new_shape = tf.cast(float_base_shape / scale, tf.int32)
        octave_shapes.append(new_shape)
    octave_shapes = octave_shapes[::-1]

    shrunk_original = tf.image.resize(img, octave_shapes[0])
    img = tf.image.resize(img, octave_shapes[0])

    for octave, shape in enumerate(octave_shapes):
        print(f"Octave {octave + 1}/{len(octave_shapes)} | shape={tuple(shape.numpy())}")
        img = tf.image.resize(img, shape)

        for step in range(steps_per_octave):
            loss, img = deepdream_step(img, step_size, model, layer_names, layer_weights_dict)
            if (step + 1) % 25 == 0:
                print(f"  Step {step + 1:>3} | Loss {loss.numpy():.4f}")

        # Reintroduce detail lost during repeated resizing.
        upscaled_shrunk_original = tf.image.resize(shrunk_original, shape)
        same_size_original = tf.image.resize(load_img(image_path, max_dim=700), shape)
        lost_detail = same_size_original - upscaled_shrunk_original
        img = img + lost_detail
        shrunk_original = tf.image.resize(load_img(image_path, max_dim=700), shape)

    return img


img = load_img(image_path, max_dim=700)
dream_img_octaves = run_deepdream_octaves(img, steps_per_octave=50, step_size=0.01, num_octaves=3)
show_tensor_image(dream_img_octaves, title="DeepDream (multi-scale / octaves)")

## 9. Compare shallow, mid, and deep layers

Using the same source image with different target layers gives a useful view of what each level of the network tends to respond to.

In [ ]:
def dream_with_layers(img, layer_weight_map, steps_per_octave=40, step_size=0.01, num_octaves=3):
    """Run DeepDream using any custom layer-to-weight mapping."""
    names = list(layer_weight_map.keys())
    outputs = [base_model.get_layer(name).output for name in names]
    model = tf.keras.Model(inputs=base_model.input, outputs=outputs)
    return run_deepdream_octaves(
        img=img,
        steps_per_octave=steps_per_octave,
        step_size=step_size,
        num_octaves=num_octaves,
        model=model,
        layer_names=names,
        layer_weights_dict=layer_weight_map,
    )


img = load_img(image_path, max_dim=700)

dream_shallow = dream_with_layers(img, {"mixed2": 1.0}, steps_per_octave=35, step_size=0.01, num_octaves=3)
dream_mid     = dream_with_layers(img, {"mixed5": 1.0}, steps_per_octave=35, step_size=0.01, num_octaves=3)
dream_deep    = dream_with_layers(img, {"mixed8": 1.0}, steps_per_octave=35, step_size=0.01, num_octaves=3)

In [ ]:
# Side-by-side comparison of the original image and three target layers.
fig, axes = plt.subplots(1, 4, figsize=(22, 7))

axes[0].imshow(Image.open(image_path).convert("RGB"))
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(deprocess(dream_shallow))
axes[1].set_title("Shallow — mixed2\n(geometry, texture, swirls)")
axes[1].axis("off")

axes[2].imshow(deprocess(dream_mid))
axes[2].set_title("Mid — mixed5\n(texture + partial shapes)")
axes[2].axis("off")

axes[3].imshow(deprocess(dream_deep))
axes[3].set_title("Deep — mixed8\n(object-like fragments)")
axes[3].axis("off")

plt.tight_layout()
plt.show()

## 10. Save generated outputs

This section writes the main results to disk so they can be reused outside the notebook.

In [ ]:
output_dir = "deepdream_outputs"
os.makedirs(output_dir, exist_ok=True)

Image.open(image_path).convert("RGB").save(os.path.join(output_dir, "original.png"))
deprocess(dream_img_simple).save(os.path.join(output_dir, "dream_simple.png"))
deprocess(dream_img_octaves).save(os.path.join(output_dir, "dream_octaves.png"))
deprocess(dream_shallow).save(os.path.join(output_dir, "dream_shallow_mixed2.png"))
deprocess(dream_mid).save(os.path.join(output_dir, "dream_mid_mixed5.png"))
deprocess(dream_deep).save(os.path.join(output_dir, "dream_deep_mixed8.png"))

print("Saved:")
for f in sorted(os.listdir(output_dir)):
    print("-", os.path.join(output_dir, f))

## 11. Save progression frames

These frames show the dream intensifying over time and can be stitched into an animation.

In [ ]:
def save_progression_frames(img, layer_weight_map, frame_count=12, steps_per_frame=8, step_size=0.01):
    """
    Save intermediate frames during optimization.

    This is useful when you want to visualize how the dream develops over time.
    """
    names = list(layer_weight_map.keys())
    outputs = [base_model.get_layer(name).output for name in names]
    model = tf.keras.Model(inputs=base_model.input, outputs=outputs)

    progress_dir = os.path.join(output_dir, "progression_frames")
    os.makedirs(progress_dir, exist_ok=True)

    frames = []
    current = tf.identity(img)
    for i in range(frame_count):
        for _ in range(steps_per_frame):
            loss, current = deepdream_step(current, step_size, model, names, layer_weight_map)
        frame = deprocess(current)
        frame.save(os.path.join(progress_dir, f"frame_{i:03d}.png"))
        frames.append(frame)

    print(f"Saved {frame_count} frames to:", progress_dir)
    return frames


img = load_img(image_path, max_dim=700)
frames = save_progression_frames(
    img, {"mixed5": 1.0, "mixed8": 1.2}, frame_count=12, steps_per_frame=10
)

## 12. Preview the progression as a GIF

The final cell renders the saved frames as a lightweight in-notebook animation.

In [ ]:
def frames_to_gif_html(frames, resize_to=(600, 400), duration=120):
    """Convert a list of PIL images into an inline looping GIF."""
    resized = [f.resize(resize_to, Image.LANCZOS) for f in frames]
    buf = io.BytesIO()
    resized[0].save(
        buf,
        format="GIF",
        save_all=True,
        append_images=resized[1:],
        loop=0,
        duration=duration,
    )
    b64 = base64.b64encode(buf.getvalue()).decode()
    return HTML(f'<img src="data:image/gif;base64,{b64}" style="border-radius:8px;"/>')

display(frames_to_gif_html(frames, resize_to=(640, 400), duration=150))